# Funnels: where do people drop off?

Practice notebook for the datafromzero SQL lesson **Funnels: where do people drop off?**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Fix the group

In [ ]:
con.sql("""
SELECT COUNT(*) AS signed_up, MIN(signup_date) AS first_signup, MAX(signup_date) AS last_signup
FROM customers
WHERE signup_date >= DATE '2026-01-01' AND signup_date <= DATE '2026-05-31'
""").show()

### Step 2. Count each customer's orders in the window

In [ ]:
con.sql("""
WITH cohort AS (
  SELECT customer_id, signup_date
  FROM customers
  WHERE signup_date >= DATE '2026-01-01' AND signup_date <= DATE '2026-05-31'
),
in_window AS (
  SELECT c.customer_id, COUNT(o.order_id) AS orders_30d
  FROM cohort c
  LEFT JOIN orders o
    ON o.customer_id = c.customer_id
   AND o.status = 'completed'
   AND CAST(o.ordered_at AS DATE) - c.signup_date <= 30
  GROUP BY c.customer_id
)
SELECT * FROM in_window
ORDER BY customer_id
LIMIT 5
""").show()

### Step 3. Check the grain

In [ ]:
con.sql("""
WITH cohort AS (
  SELECT customer_id, signup_date
  FROM customers
  WHERE signup_date >= DATE '2026-01-01' AND signup_date <= DATE '2026-05-31'
),
in_window AS (
  SELECT c.customer_id, COUNT(o.order_id) AS orders_30d
  FROM cohort c
  LEFT JOIN orders o
    ON o.customer_id = c.customer_id
   AND o.status = 'completed'
   AND CAST(o.ordered_at AS DATE) - c.signup_date <= 30
  GROUP BY c.customer_id
)
SELECT COUNT(*) AS customers, SUM(orders_30d) AS orders
FROM in_window
""").show()

### Step 4. Count the customers at each step

In [ ]:
con.sql("""
WITH cohort AS (
  SELECT customer_id, signup_date
  FROM customers
  WHERE signup_date >= DATE '2026-01-01' AND signup_date <= DATE '2026-05-31'
),
in_window AS (
  SELECT c.customer_id, COUNT(o.order_id) AS orders_30d
  FROM cohort c
  LEFT JOIN orders o
    ON o.customer_id = c.customer_id
   AND o.status = 'completed'
   AND CAST(o.ordered_at AS DATE) - c.signup_date <= 30
  GROUP BY c.customer_id
)
SELECT COUNT(*) AS signed_up,
       COUNT(CASE WHEN orders_30d >= 1 THEN 1 END) AS first_order,
       COUNT(CASE WHEN orders_30d >= 2 THEN 1 END) AS second_order,
       COUNT(CASE WHEN orders_30d >= 3 THEN 1 END) AS third_order
FROM in_window
""").show()

### Step 5. Add the conversion

In [ ]:
con.sql("""
WITH cohort AS (
  SELECT customer_id, signup_date
  FROM customers
  WHERE signup_date >= DATE '2026-01-01' AND signup_date <= DATE '2026-05-31'
),
in_window AS (
  SELECT c.customer_id, COUNT(o.order_id) AS orders_30d
  FROM cohort c
  LEFT JOIN orders o
    ON o.customer_id = c.customer_id
   AND o.status = 'completed'
   AND CAST(o.ordered_at AS DATE) - c.signup_date <= 30
  GROUP BY c.customer_id
),
steps AS (
  SELECT COUNT(*) AS signed_up,
         COUNT(CASE WHEN orders_30d >= 1 THEN 1 END) AS first_order,
         COUNT(CASE WHEN orders_30d >= 2 THEN 1 END) AS second_order,
         COUNT(CASE WHEN orders_30d >= 3 THEN 1 END) AS third_order
  FROM in_window
)
SELECT *,
       ROUND(100.0 * first_order / signed_up, 1) AS conv_1,
       ROUND(100.0 * second_order / first_order, 1) AS conv_2,
       ROUND(100.0 * third_order / second_order, 1) AS conv_3,
       ROUND(100.0 * third_order / signed_up, 1) AS overall
FROM steps
""").show()

### Step 6. Verify: count the orders in the window directly

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders_in_window
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
WHERE c.signup_date >= DATE '2026-01-01' AND c.signup_date <= DATE '2026-05-31'
  AND o.status = 'completed'
  AND CAST(o.ordered_at AS DATE) - c.signup_date <= 30
""").show()

## Exercise 1. The funnel with a 7-day window

Fill in the two numbers marked `...`: the last signup date that has a full window in the data, and the number of days.
Expected: 1 row, 4 columns.

In [ ]:
con.sql("""
-- replace the two ... below before you run it
WITH cohort AS (
  SELECT customer_id, signup_date
  FROM customers
  WHERE signup_date >= DATE '2026-01-01' AND signup_date <= DATE '...'
),
in_window AS (
  SELECT c.customer_id, COUNT(o.order_id) AS orders_7d
  FROM cohort c
  LEFT JOIN orders o
    ON o.customer_id = c.customer_id
   AND o.status = 'completed'
   AND CAST(o.ordered_at AS DATE) - c.signup_date <= ...
  GROUP BY c.customer_id
)
SELECT COUNT(*) AS signed_up,
       COUNT(CASE WHEN orders_7d >= 1 THEN 1 END) AS first_order,
       COUNT(CASE WHEN orders_7d >= 2 THEN 1 END) AS second_order,
       COUNT(CASE WHEN orders_7d >= 3 THEN 1 END) AS third_order
FROM in_window
""").show()

## Exercise 2. Does the funnel differ by loyalty tier?

Use the 30-day window and the 49 customers of the worked example. Before you run anything, write one sentence in the cell
below: what one row of the result is, and how many customers you expect in the smallest tier. Then add `loyalty_tier` to
the cohort and group the last step by it.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Query: the funnel by loyalty tier

## Exercise 3. Spot the flaw

A colleague builds the funnel for every customer who signed up in 2026 and reports that only 41.5% reach a third order.
Run the query, then find out what went wrong and write the fair version.

In [ ]:
con.sql("""
WITH cohort AS (
  SELECT customer_id, signup_date
  FROM customers
  WHERE signup_date >= DATE '2026-01-01'
),
in_window AS (
  SELECT c.customer_id, COUNT(o.order_id) AS orders_30d
  FROM cohort c
  LEFT JOIN orders o
    ON o.customer_id = c.customer_id
   AND o.status = 'completed'
   AND CAST(o.ordered_at AS DATE) - c.signup_date <= 30
  GROUP BY c.customer_id
)
SELECT COUNT(*) AS signed_up,
       COUNT(CASE WHEN orders_30d >= 1 THEN 1 END) AS first_order,
       COUNT(CASE WHEN orders_30d >= 2 THEN 1 END) AS second_order,
       COUNT(CASE WHEN orders_30d >= 3 THEN 1 END) AS third_order
FROM in_window
""").show()

In [ ]:
# Your corrected query: